# Part B - Low-Rank Adaptation (LoRA)
## TOTAL MARKS: 50
## Overview
### `Name: Muhammad Anas Naveed`
### `Roll Number: 28100045`

References:
https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch


## Instructions

<table style="width:100%; table-layout:fixed;">
<tr>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Plagiarism Policy
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>All work must be done independently.</strong><br>
      Any plagiarism or cheating (from peers or the internet) will be immediately referred to the DC.
      If you are unsure what constitutes plagiarism, consult the TAs in a timely manner.
    </li>
    <li style="margin-top:8px;">
      <strong>Do not look at anyone else’s code.</strong>
    </li>
  </ul>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    Submission Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      Submit <strong>all</strong> required <code>.ipynb</code> and <code>.py</code> files on LMS
      (search how to extract scripts if confused).
    </li>
    <li style="margin-top:8px;">
      <strong>Submissions via Dropbox or email will not be accepted.</strong>
    </li>
    <li style="margin-top:8px;">
      Zip your files as <code>RollNumber_PAx.zip</code> and ensure your roll number is filled in correctly.
    </li>
    <li style="margin-top:8px;">
      <strong>Deviation from the naming convention will result in a penalty.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Expected file structure:</strong>
    </li>
  </ul>
  <pre style="margin:8px 0 0 0; font-size:90%; color:#e53935;">
26100076_PA3.zip
├─ 26100076_PA3.ipynb
└─ 26100076_PA3.py
  </pre>
</td>

<td style="vertical-align:top; padding:12px; width:33%;">
  <h3 style="margin:0 0 8px 0; font-weight:800; color:#e53935;">
    General Instructions
  </h3>
  <ul style="margin:6px 0 0 18px; padding:0; color:#e53935;">
    <li>
      <strong>Ensure all cells are executed before submission.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>Do not remove or modify any pre-written code.</strong>
    </li>
    <li style="margin-top:8px;">
      <strong>All parts of the assignment must be attempted.</strong>
    </li>
  </ul>
</td>

</tr>
</table>

# Low-Rank Adaptation (LoRA) — An Intuitive Explanation

Large language models (LLMs) and vision transformers contain **millions or billions of parameters** (weights).  
Fine-tuning these models normally requires updating *all* of these weights, which is computationally expensive and often limited by GPU memory.

---

### Regular Fine-Tuning

In standard training or fine-tuning:

- Each layer has a large weight matrix **W**
- Training learns a full update matrix **ΔW**
- The updated weights are:

$$
W_{\text{updated}} = W + \Delta W
$$

**Drawback:**  
The update matrix **ΔW** is very large, making training slow and memory-intensive.

---

### LoRA: The Key Idea

LoRA (Low-Rank Adaptation) is based on the observation that:

> Most useful weight updates can be represented by **simple patterns**, rather than a full, complex matrix.

Instead of learning the full **ΔW**, LoRA **approximates it** using two much smaller matrices:

$$
\Delta W \approx A \cdot B
$$

where:
- **A** and **B** are small matrices
- Their matrix product has the same shape as **W**

The weight update becomes:

$$
W_{\text{updated}} = W + A \cdot B
$$

The figure below illustrates these formulas for full finetuning and LoRA side by side.

![LoRA illustration](LoRAImage.webp)

---

### Intuitive Explanation

Think of the model as a large machine with many adjustment knobs.

- **Full fine-tuning:**  
  Adjust every knob individually.

- **LoRA:**  
  Keep the original knobs fixed and add a small attachment that adjusts many knobs together in a coordinated way.

This attachment (matrices **A** and **B**) captures the most important changes while using far fewer parameters.

## Implementing a LoRA Layer (2.5 MARKS)
Implement a LoRA layer that adapts a neural network without directly modifying its original weights. Instead of learning a full weight update, represent the adaptation as a low-rank decomposition that captures task-specific changes using a compact set of parameters. This formulation reduces both memory usage and computational cost compared to full fine-tuning.

Design the LoRA layer using two trainable low-rank matrices whose product defines the weight update. Scale this update appropriately and optionally apply dropout to regulate its effect. During the forward pass, compute the low-rank update independently so it can later be combined with the output of a frozen base layer in a modular and efficient manner.

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [20]:
class LoRALayer(nn.Module):
    """
    Implements a LoRA low-rank adaptation layer.
    """

    def __init__(self, in_features, out_features = None, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        # TODO: store rank and alpha
        if isinstance(in_features, nn.Linear):
            linear = in_features
            in_features = linear.in_features
            out_features = linear.out_features
        self.rank = rank
        self.scaling = alpha / rank
        # TODO: initialize A and B matrices
        self.A = nn.Parameter(torch.randn(in_features, rank))
        self.B = nn.Parameter(torch.zeros(rank, out_features))
        # TODO: initialize dropout
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # TODO: apply dropout
        x = self.dropout(x)
        # TODO: compute low-rank update
        lora_update = x @ self.A @ self.B
        # TODO: scale and return
        return lora_update * self.scaling

## LoRA-Adapted Linear Layer (non-merged) (5 MARKS)
Implement a LoRA-adapted version of a linear layer by wrapping an existing nn.Linear module. The original weights and bias should be frozen to ensure that only the low-rank adaptation is trained. The forward pass should combine the output of the frozen linear layer with the output of the LoRA layer, illustrating how task-specific updates can be added without modifying the base parameters.

In [21]:
class LoRALinear(nn.Module):
    """
    A linear layer with LoRA adaptation.
    """

    def __init__(self, linear_layer, rank=8, alpha=16, dropout=0.0):
        super().__init__()
        # TODO: store linear layer
        self.linear = linear_layer
        # TODO: freeze base weights
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False
        # TODO: create LoRALayer
        self.lora = LoRALayer(linear_layer.in_features, linear_layer.out_features, rank, alpha, dropout)
        self.lora = self.lora.to(linear_layer.weight.device)

    def forward(self, x):
        # TODO: compute base output
        base_out = self.linear(x)
        # TODO: compute LoRA output
        lora_out = self.lora(x)
        # TODO: return combined output
        return base_out + lora_out


## Define a Small Network with One Linear Layer (2.5 MARKS)
Create a simple experimental setup using a single linear layer and a randomly generated input tensor. Fix the random seed to ensure reproducibility. Compute and record the output of the original linear layer, which will serve as a baseline for comparison when LoRA is introduced.

In [22]:
# TODO: set random seed
random_seed = 42
torch.manual_seed(random_seed)

# TODO: create linear layer
layer = nn.Linear(10, 5)
# TODO: create input tensor
x = torch.randn(4, 10)
# TODO: compute and print baseline output
baseline_output = layer(x)
print("Baseline output:\n", baseline_output)


Baseline output:
 tensor([[ 0.7910, -0.6975,  0.4384,  0.7299,  1.0319],
        [-0.2977,  0.5749, -0.3397,  0.9044,  1.0887],
        [-0.3758, -0.0150, -0.1559, -0.4732,  0.4536],
        [ 0.3639,  0.1870, -0.7518,  0.3870,  0.4043]],
       grad_fn=<AddmmBackward0>)


## Apply LoRA to the Linear Layer (2.5 MARKS)
Apply the LoRA-adapted linear wrapper to the previously defined layer. Run the same input through this modified layer and observe the change in output compared to the baseline. This step demonstrates how LoRA introduces an additional learned transformation while keeping the original layer weights unchanged.

In [23]:
# TODO: wrap layer with LoRALinear
# TODO: compute and print LoRA output
layer_lora = LoRALayer(layer,rank=2,alpha=4)
lora_output = LoRALinear(layer, rank=2, alpha=4)(x)
print("LoRA output:\n", lora_output)

LoRA output:
 tensor([[ 0.7910, -0.6975,  0.4384,  0.7299,  1.0319],
        [-0.2977,  0.5749, -0.3397,  0.9044,  1.0887],
        [-0.3758, -0.0150, -0.1559, -0.4732,  0.4536],
        [ 0.3639,  0.1870, -0.7518,  0.3870,  0.4043]], grad_fn=<AddBackward0>)


## Merged LoRA Version (for inference) (5 MARKS)
Implement an alternative version of the linear layer in which the LoRA weight update is merged directly into the original weight matrix. In this formulation, compute the low-rank weight update and add it to the frozen base weights before applying the linear operation. This merged approach reflects how LoRA can be deployed efficiently during inference.

In [24]:
class LinearWithLoRAMerged(nn.Module):
    """
    Linear layer where LoRA weights are merged into W.
    """

    def __init__(self, linear, rank, alpha):
        super().__init__()
        # TODO: store linear layer
        self.linear = linear
        # TODO: create LoRALayer
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)

    def forward(self, x):
        # TODO: compute delta W
        delta_W = (self.lora.A @ self.lora.B).T * self.lora.scaling
        # TODO: merge weights
        merged_weight = self.linear.weight + delta_W
        # TODO: apply linear op
        return F.linear(x, merged_weight, self.linear.bias)


## Verify Mathematical Equivalence  (0.5 MARKS)
Verify that the merged and non-merged LoRA implementations produce identical outputs when initialized with the same parameters. Ensure both models share the same low-rank matrices, then compare their outputs on the same input. Report the maximum absolute difference to confirm numerical equivalence between the two approaches.

In [25]:
# TODO: instantiate merged and unmerged models
# TODO: copy LoRA parameters
# TODO: compare outputs

layer_lora_unmerged = LoRALinear(layer, rank=2, alpha=4)
layer_lora_merged = LinearWithLoRAMerged(layer, rank=2, alpha=4)

layer_lora_merged.lora.A.data = layer_lora_unmerged.lora.A.data.clone()
layer_lora_merged.lora.B.data = layer_lora_unmerged.lora.B.data.clone()

out_unmerged = layer_lora_unmerged(x)
out_merged = layer_lora_merged(x)
max_diff = (out_unmerged - out_merged).abs().max().item()
print(f"Max absolute difference: {max_diff}")




Max absolute difference: 0.0


# Applying LoRA Layers to LLM (2.5 MARKS)


## Preparing the Dataset
Prepare a text dataset suitable for causal language modeling. Load a subset of the AG News dataset, shuffle it for reproducibility, and split it into training and evaluation sets. Tokenize the text with fixed-length padding and truncation, and construct labels that mask padding tokens so they do not contribute to the training loss. Finally, format the dataset for PyTorch and create data loaders for batch-based training and evaluation.

In [26]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset


In [27]:
from datasets import load_dataset
from torch.utils.data import DataLoader

def prepare_dataset(tokenizer, max_samples=500, max_length=128):
    """
    Prepares train and test DataLoaders for language model fine-tuning.

    Args:
        tokenizer: HuggingFace tokenizer
        max_samples: number of training samples to use
        max_length: maximum sequence length

    Returns:
        train_loader, test_loader
    """
    dataset = load_dataset("ag_news", split="train")
    dataset = dataset.shuffle(seed=42)
    train_data = dataset.select(range(max_samples))
    test_data = dataset.select(range(max_samples, max_samples + 100))

    def tokenize(batch):
        encoding = tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=max_length
        )
        labels = []
        for ids, mask in zip(encoding["input_ids"], encoding["attention_mask"]):
            label = [token if m == 1 else -100 for token, m in zip(ids, mask)]
            labels.append(label)
        encoding["labels"] = labels
        return encoding

    train_data = train_data.map(tokenize, batched=True, remove_columns=train_data.column_names)
    test_data = test_data.map(tokenize, batched=True, remove_columns=test_data.column_names)

    train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    train_loader = DataLoader(train_data, batch_size=8, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=8)

    return train_loader, test_loader

## Initializing the Base Language Model
Load a pre-trained causal language model and its corresponding tokenizer. Configure the tokenizer to use a valid padding token and move the model to the appropriate computation device. Keep this base model in evaluation mode to serve as a frozen reference for comparison with the LoRA-adapted version.

In [28]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [29]:
# TODO: load tokenizer
# TODO: load base model
model_name = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
# TODO: set device and eval mode
model.eval()


Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Creating a LoRA-Adapted Model (2.5 MARKS)
Instantiate a second copy of the pre-trained language model that will be modified using LoRA. This model will be used for training and should be set to training mode. Maintaining separate base and LoRA-adapted models allows direct comparison of parameter counts and downstream performance.

In [30]:
# TODO: load LoRA model
# TODO: set training mode
model_lora = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
model_lora.train()

Loading weights:   0%|          | 0/160 [00:00<?, ?it/s]

GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-125M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 2, 4, 6, 8, 10}.attn.attention.bias | UNEXPECTED |  | 
transformer.h.{0...11}.attn.attention.masked_bias     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=False)
            (q_proj): Linear(in_features=768, out_features=768, bias=False)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPTNeoMLP(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (c_proj): Linear(in_fe

## Applying LoRA to Attention Layers (2.5 MARKS)
Modify the language model by replacing selected linear layers within the attention mechanism with LoRA-adapted linear layers. Target projection layers commonly used in self-attention, such as query, key, value, and output projections. This step injects low-rank adaptations into the model while leaving the original weights structurally intact.

In [31]:
def apply_lora_to_model(model, rank=8, alpha=16):
    """
    Replaces selected attention projection Linear layers with LoRALinear wrappers.
    Args:
        model: a transformer model (HuggingFace or similar)
        rank: LoRA rank r
        alpha: LoRA scaling alpha
    Returns:
        model (modified in-place or returned for convenience)
    """
    target_names = {"q_proj", "k_proj", "v_proj", "out_proj"}
    
    for parent_name, parent_module in model.named_modules():
        for child_name, child_module in list(parent_module.named_children()):
            if child_name in target_names and isinstance(child_module, nn.Linear):
                setattr(parent_module, child_name, LoRALinear(child_module, rank, alpha))
    
    return model

In [32]:
apply_lora_to_model(model_lora, rank=8, alpha=16)


GPTNeoForCausalLM(
  (transformer): GPTNeoModel(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(2048, 768)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPTNeoBlock(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPTNeoAttention(
          (attention): GPTNeoSelfAttention(
            (attn_dropout): Dropout(p=0.0, inplace=False)
            (resid_dropout): Dropout(p=0.0, inplace=False)
            (k_proj): LoRALinear(
              (linear): Linear(in_features=768, out_features=768, bias=False)
              (lora): LoRALayer(
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (v_proj): LoRALinear(
              (linear): Linear(in_features=768, out_features=768, bias=False)
              (lora): LoRALayer(
                (dropout): Dropout(p=0.0, inplace=False)
              )
            )
            (q_proj): LoRALinear(
              (linear): Linea

## Freezing Base Parameters and Enabling LoRA Training (2.5 MARKS)
Freeze all original model parameters to prevent them from being updated during training. Enable gradient updates only for the LoRA low-rank matrices. This ensures that learning is restricted to the parameter-efficient LoRA components while the pre-trained knowledge of the base model is preserved.

In [33]:
# TODO: freeze base parameters
# TODO: enable LoRA parameters
for name, param in model_lora.named_parameters():
    if "lora" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

## Inspecting Trainable Parameters
Verify that only the intended LoRA parameters are trainable by inspecting the model’s parameter list. This step helps confirm that the parameter-freezing strategy has been applied correctly before training begins.

In [34]:
def check_trainable_params(model):
    # TODO: print trainable parameters
    for name, param in model.named_parameters():
        if param.requires_grad:
            print(name)

check_trainable_params(model_lora)

transformer.h.0.attn.attention.k_proj.lora.A
transformer.h.0.attn.attention.k_proj.lora.B
transformer.h.0.attn.attention.v_proj.lora.A
transformer.h.0.attn.attention.v_proj.lora.B
transformer.h.0.attn.attention.q_proj.lora.A
transformer.h.0.attn.attention.q_proj.lora.B
transformer.h.0.attn.attention.out_proj.lora.A
transformer.h.0.attn.attention.out_proj.lora.B
transformer.h.1.attn.attention.k_proj.lora.A
transformer.h.1.attn.attention.k_proj.lora.B
transformer.h.1.attn.attention.v_proj.lora.A
transformer.h.1.attn.attention.v_proj.lora.B
transformer.h.1.attn.attention.q_proj.lora.A
transformer.h.1.attn.attention.q_proj.lora.B
transformer.h.1.attn.attention.out_proj.lora.A
transformer.h.1.attn.attention.out_proj.lora.B
transformer.h.2.attn.attention.k_proj.lora.A
transformer.h.2.attn.attention.k_proj.lora.B
transformer.h.2.attn.attention.v_proj.lora.A
transformer.h.2.attn.attention.v_proj.lora.B
transformer.h.2.attn.attention.q_proj.lora.A
transformer.h.2.attn.attention.q_proj.lora.B
tr

## Comparing Parameter Counts  (2.5 MARKS)
Compute and compare the total number of parameters and the number of trainable parameters for both the base model and the LoRA-adapted model. Report the percentage reduction in trainable parameters achieved through LoRA to quantify its efficiency benefits.

In [35]:
def count_parameters(model):
    # TODO: compute parameter counts
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters:     {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"Trainable percentage: {100 * trainable / total:.4f}%")

print("Base model:")
count_parameters(model)
print("\nLoRA model:")
count_parameters(model_lora)

Base model:
Total parameters:     125,198,592
Trainable parameters: 125,198,592
Trainable percentage: 100.0000%

LoRA model:
Total parameters:     125,788,416
Trainable parameters: 589,824
Trainable percentage: 0.4689%


## Training the LoRA-Adapted Model  (2.5 MARKS)
Train the LoRA-adapted model using the prepared dataset and an appropriate optimizer. Perform multiple training epochs while monitoring the training loss. Since only a small subset of parameters is being updated, this training process is significantly more efficient than full fine-tuning.

In [36]:
# TODO: training loop
train_loader, test_loader = prepare_dataset(tokenizer)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model_lora.parameters()), lr=3e-4
)

model_lora.train()

for epoch in range(2):
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model_lora(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        if step % 50 == 0:
            print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")

## DO NOT FORGET TO PRINT YOUR LOSS

Epoch 0 | Step 0 | Loss: 4.3764
Epoch 0 | Step 50 | Loss: 4.4863
Epoch 1 | Step 0 | Loss: 3.7512
Epoch 1 | Step 50 | Loss: 3.6735


## Evaluating Model Accuracy  (5 MARKS)
Evaluate both the original base model and the LoRA-adapted model on the test dataset. Compute token-level accuracy while ignoring masked positions. Compare the results to assess how well the LoRA-based fine-tuning performs relative to the frozen pre-trained model.

In [41]:
@torch.no_grad()
def compute_accuracy(model, dataloader, device):
    """
    TODO:
    1. Disable gradients and set model to eval mode
    2. Get model logits for each batch
    3. Compute predictions using argmax
    4. Mask out padding tokens (-100)
    5. Return token-level accuracy (%)
    """
    model.eval()
    correct = 0
    total = 0

    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits  # (B, T, V)

        # shift: predictions from t=0..T-2, labels from t=1..T-1
        shift_logits = logits[:, :-1, :]
        shift_labels = batch["labels"][:, 1:]

        preds = shift_logits.argmax(dim=-1)

        mask = shift_labels != -100
        correct += (preds[mask] == shift_labels[mask]).sum().item()
        total += mask.sum().item()

    return (correct / total) * 100 if total > 0 else 0.0

In [42]:
print(f"Test accuracy orig model: {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA model: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

Test accuracy orig model: 29.81%
Test accuracy LoRA model: 34.40%


# Reflective Questions (12 Marks)
Answer the reflective questions provided below.

Q1. Why is representing weight updates as a low-rank decomposition effective for large language models? \
Ans. Representing the weight updates as a low-rank decomposition reduces the optimizer memory required during fine tuning.As the weight updates have a low intinsic rank, meaning more information can be captured in a smaller subspace. Initially, the size of the weight update matrix is d x d but LoRA decomoposes the matrix into two smaller matrices B and A  of shapes dx r and r x d where r <= d, resuding the trainable parameters. At inference, BA is added to the frozen weights with no added latency. Different LoRA adapters can also be swapped to serve different tasks without reforming the full model.

Q2. What assumptions does LoRA make about the nature of task-specific adaptations?\
Ans. LoRa assumes that the Pre-Trained Frozen parameters has already captured rich general information and just minor tweaks/corrections are needed rather than full reparameterization. It also assumes that the important changes to information needed to adapt the pre tained model for the new task can be captured by small number of adjustments in the weight space and that the task specific adaptation lie in a low-dimensional subspace rathen than full parameter space.

Q3. Why are LoRA layers typically applied to attention projection layers rather than all linear layers?\
Ans. The attention projection layers are most task sensitive and crucial part of the model architecture as here it learns which parts of the input to focus on. Applying it to all the layers including the FF-layers only drastically increases the number of trainable parameters without much gains. Targeting the attention projection layers give the best tradeoff between computational cost and task-adaptation.

Q4. How does the choice of rank affect the expressive capacity of the LoRA adaptation?\
Ans. A higher rank results in a increase in the models capability to capture complex information/ adaptations at the cost of more tainable parameters. On the other hand, decreasing the ranks reduces the number of trainable parameters but may not be able to grasp the task-adapatations that cannot be represented in a low dimensional subspace. In the end,too low and the model cannot adapt sufficiently, too high and the efficiency benefit diminishes.

Q5. Why might a LoRA-adapted model perform comparably to a fully fine-tuned model on some tasks?\
Ans. The LLMS are pretrained on vast amount of data due to which their parameters contain broad linguistic knowledge, that given few directions helps the model fine tune to a newer task. If the intrinsic dimensionality of the task-specific update is genuinely low, then LoRA's low-rank approximation loses very little information compared to a full update, resulting in comparable performance with far fewer parameters.

Q6. Under what conditions could LoRA lead to worse performance than full fine-tuning?
Ans. LoRA compresses the task-adaptiopns into a low dimensional subspace, but if the adaptations are too complex too be project onto a compact subspace crucial information may be lost. It would also struggle with task requiring major changes to the original weights as LoRA keeps them frozen and only do minor changes. So if the pretained model is too small to have rich representations then LoRA's constrained update becomes a bottleneck.
